# TP 2 : Branch-and-bound applied to a knapsack problem

# Récupération des données

In [ ]:
function readKnaptxtInstance(filename)
    price=Int64[]
    weight=Int64[]
    KnapCap=Int64[]
    open(filename) do f
        for i in 1:3
            tok = split(readline(f))
            if(tok[1] == "ListPrices=")
                for i in 2:(length(tok)-1)
                    push!(price,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "ListWeights=")
                for i in 2:(length(tok)-1)
                    push!(weight,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "Capacity=")
                push!(KnapCap, parse(Int64, tok[2]))
            else
                println("Unknown read :", tok)
            end
        end
    end
    capacity=KnapCap[1]
    return price, weight, capacity
end

# Calcul des bornes supérieures

In [ ]:
#on remplie le sac avec l'objet de meilleur ratio
function borne1((sac, free_capacity, valeur), price, weight, capacity) 
    ratio = [price[i]/weight[i] for i in 1:length(price)]

    # On trouve l'objet de meilleur ratio encore undéfini
    best_ratio = -1
    best_ratio_index = 1
    for i in 1:length(ratio)
        if sac[i] == -1 && ratio[i] > best_ratio
            best_ratio = ratio[i]
            best_ratio_index = i
        end
    end

    return valeur + best_ratio * free_capacity
end

#calcul de la borne supérieure du problème du sac à dos fractionnaire
function borne2((sac, free_capacity, valeur), price, weight, capacity)
    ratio = [price[i]/weight[i] for i in 1:length(price)]
    s = copy(sac)
    cap = free_capacity
    val = valeur
    while cap > 0 && any(x -> x == -1, s)
        # On trouve l'objet de meilleur ratio encore undéfini
        best_ratio = -1
        best_ratio_index = 1
        for i in 1:length(ratio)
            if s[i] == -1 && ratio[i] > best_ratio
                best_ratio = ratio[i]
                best_ratio_index = i
            end
        end
        # On ajoute ce qu'on peut de l'objet dans le sac
        if cap >= weight[best_ratio_index]
            s[best_ratio_index] = 1
            cap -= weight[best_ratio_index]
            val += price[best_ratio_index]
        else
            s[best_ratio_index] = cap / weight[best_ratio_index]
            val += price[best_ratio_index] * cap / weight[best_ratio_index]
            cap = 0
        end
    end
    return val
end

# Test de sondabilité

In [ ]:
function testSondability_LP((sac, free_capacity, valeur), price, weight, capacity, borne)
    TA, TO, TR = false, false, false
    global Bestsol
    global BestProfit

    borne_sup = 0
    if borne == 1
        borne_sup = borne1((sac, free_capacity, valeur), price, weight, capacity)
    elseif borne == 2
        borne_sup = borne2((sac, free_capacity, valeur), price, weight, capacity)
    end

    if(free_capacity < 0)#Test de faisabilite
        TA=true

    elseif(all(x -> x > -1, sac)) #Test de resolution
        TR=true
        #if (value(benef) >= BestProfit)
        if (valeur >= BestProfit)
            Bestsol = sac
            #BestProfit=value(benef)
            BestProfit=valeur
        end

    elseif(borne_sup < BestProfit) #Test d'optimalite
        TO=true

    else
    end
    TA, TO, TR
end

# Création du modèle et boucle principale

In [ ]:
function createModel_LP(price, weight, capacity)
# ROOT NODE
    n=length(price)

    sac = [-1. for i in 1:n] 
    free_capacity = capacity 
    valeur = 0

    return (sac, free_capacity, valeur)
end

function solver(model, price, weight, capacity, borne)
    sac = model[1]
    TA, TO, TR= testSondability_LP(model, price, weight, capacity, borne)
    if !(TA || TO || TR)
        ratio = [price[i]/weight[i] for i in 1:length(price)]
        best_ratio = -1
        best_ratio_index = 0
        for i in 1:length(ratio)
            if (sac[i] == -1 && ratio[i] > best_ratio)
                best_ratio = ratio[i]
                best_ratio_index = i
            end
        end
        sac1 = copy(model[1])
        sac1[best_ratio_index] = 1
        model1  = (sac1, model[2] - weight[best_ratio_index], model[3] + price[best_ratio_index])
        solver(model1, price, weight, capacity, borne)
        sac2 = copy(model[1])
        sac2[best_ratio_index] = 0
        model2  = (sac2, model[2], model[3])
        solver(model2, price, weight, capacity, borne)
    end
end

function solveKnapInstance(filename, borne)
    global Bestsol
    global BestProfit

    price, weight, capacity = readKnaptxtInstance(filename)

    (sac, free_capacity, valeur) = createModel_LP(price, weight, capacity)

    solver((sac, free_capacity, valeur), price, weight, capacity, borne)
    #transformer Bestsol en vecteur def indices des 1 de Bestsol
    Bestsol = [i for i in 1:length(Bestsol) if Bestsol[i] == 1]
end

function solveNdisplayKnap(filename, borne)

    println("\n Branch-and-Bound for solving a knapsack problem. \n\n Solving instance '" * filename * "'\n")

    solveKnapInstance(filename, borne)

    println("\n******\n\nOptimal value = ", BestProfit, "\n\nOptimal x=", Bestsol)

end

# Test

In [ ]:
#INSTANCE = "InstancesKnapSack/test.opb.txt"
#INSTANCE = "InstancesKnapSack/circle/knapPI_16_20_1000_1_-2291.opb.txt"
#INSTANCE = "InstancesKnapSack/strongly_correlated_span/knapPI_13_50_1000_1_-1989.opb.txt"
#INSTANCE = "InstancesKnapSack/similar_weights/knapPI_9_50_1000_3_970.opb.txt"
INSTANCE = "InstancesKnapSack/uncorrelated/knapPI_1_100_1000_2_8600.opb.txt"
global BestProfit=-1.0
global Bestsol=Float64[]
solveNdisplayKnap(INSTANCE,1)


# Questions

## Questions préliminaires

Quelle est la règle de séparation choisie ?

- On sépare sur l'objet avec le ratio $r\ =\ \frac{prix}{poids}$ le plus élevé.

Quelle méthode de calcul de borne supérieure est utilisée ?

- La borne supérieure utilisée est la borne 2, les variables étant continues.

Quels sont les tests de sondabilité TA, TO, TR ?

1) Le test TA: c'est le test d'admissibilité, cela test si le noeud actuel est valide ou non. Dans notre cas, cela veut dire que la capacité du sac est supérieure au poids total des objets mis dans le sac.

2) Le test TO: c'est le test d'optimalité, cela test si le noeud actuel peut contenir la solution optimale ou non. Cela consiste en la comparaison entre une solution connue et la meilleure solution ( borne_sup ) que ce noeud peut fournir.

3) Le test TR: c'est le test de résolution, cela test si le noeud actuel est une feuille. Dans notre cas, cela consiste en la vérification de la détermination des objets que l'on met dans le sac. Si ce test passe, on compare la valeur du sac et la solution optimale connue. Si la valeur du sac est supérieure à la solution optimale, le sac devient alors la nouvelle solution optimale connue.

Quelle est la stratégie d’exploration choisie ?

- On regarde la solution optimale du sous-arbre à gauche puis du sous-arbre à droite du noeud pour chaque noeud de l'arbre.

## Code et annalyse

Donner les points clés de votre implémentation des différents blocs du Branch-and-Bound (borne supérieure, règle de séparation, TA, TO, TR, stratégie d’exploration) et expliquer votre choix de structure de données permettant de garder les informations nécessaires au Branch-and-Bound.

- Notre modèle est constitué d'un sac, de la capacité réstante du sac et de la valeur du sac. Le sac est une liste de la taille de la liste des objets tels que l'élément vaut -1 si la présence de l'objet dans le sac est indétérminé, 0 si l'objet n'est pas dans le sac et 1 si l'objet est dans le sac. IL y a également deux variables globales que sont le profit optimale (BestProfit) et la solution optimale (Bestsol). Nous effectuons la séparation sur l'objet dont la présence dans le sac est indéterminé avec le plus grand ratio $r\ =\ \frac{prix}{poids}$. Pour le test de sondabilité, nous regardons si la capacité restante du modèle est positive pour TA, si la borne_sup de notre modèle est supérieure à la solution optimale pour TO et si il y a des éléménts du sac à -1 pour TR. Enfin, si aucun des tests TR,TO, TA ne passe, alors le noeud est sondable, dans ce cas on sépare et on rappelle le solver sur les deux sous-problèmes. Si le test TR passe, alors nous regadons si la solution est meilleure que la solution optimale, si c'est le cas la solution optimale devient la solution du noeud actuel.

Comparer les performances du Branch-and-Bound en utilisant la borne 1 ou la borne 2. Le résultat vous semble-t-il cohérent ? Argumenter. Remarque : la comparaison doit utiliser suffisamment d’instances et de tailles variées.

- Pour un petit nombre d'objet, le branch-and-bound avec la borne 1 est plus efficace que la borne 2. Cela correspond au fait que le calcul de la borne 1 est plus rapide que le calcul de la borne 2.

- Pour un grand nombre d'objet, le branch-and-bounf avec la borne 1 est moins efficace que la borne 2. Cela correspond au fait que la borne 2 est plus réaliste et plus proche de la réalité que la borne 1 permettant ainsi un meilleur test d'optimalité.

- En conclusion, le choix de la borne à utiliser dépend du nombre d'objet dont on dispose et de la capacité du sac. Il faut alors comparer le temps perdu à calculer les bornes supérieures avec la possibilité de supprimer des sous-arbres. Cependant la borne 2 devient très rapidement plus efficace que la borne 1.

Proposer des solutions pour améliorer le Branch and Bound (modifications des règles de séparation, modification des règles d’exploration, résolution de la relaxation...).

- Pour améliorer le branch-and-bound nous pouvons : 
1) modifier les règles de séparations : On rempli le sac avec les objets de plus en moins rentables, on fait ensuite la séparation sur l'objet que nous n'avons pas pu rentrer dans le sac intégralement.

2) modifier les règles d'exploration : On modifie la solution optimale étant la meilleure solution d'un noeud éyant passsé le test de résolution par la meilleure borne inférieure connue. Ainsi en alternant un parcour en largeur et en profondeur de l'arbre. Nous pourrions améliorer la suppression des sous-problème passant le test d'optimalité.

